### CREDIT RISK PREDICTION

#### STEP 1: Import Libraries

In [83]:
import pandas as pd

pd.set_option("display.max_columns", None)

#### STEP 2: Load Data

In [84]:
# Load dataset
df = pd.read_csv("../data/raw/train.csv")
dl = pd.read_csv("../data/raw/economic_indicators.csv")

df.head()

,ID,customer_id,country_id,tbl_loan_id,lender_id,loan_type,Total_Amount,Total_Amount_to_Repay,disbursement_date,due_date,duration,New_versus_Repeat,Amount_Funded_By_Lender,Lender_portion_Funded,Lender_portion_to_be_repaid,target
0,ID_257244237466267278,257244,Kenya,237466,267278,Type_1,16477.0,16642.0,2022-08-15,2022-08-22,7,Repeat Loan,2436.0,0.147842,2460.0,0
1,ID_249619214444267278,249619,Kenya,214444,267278,Type_1,5599.0,5599.0,2022-07-13,2022-07-20,7,Repeat Loan,2799.5,0.500000,2800.0,0
2,ID_251457257838267278,251457,Kenya,257838,267278,Type_1,4679.0,4746.0,2022-09-12,2022-09-19,7,Repeat Loan,7.8,0.001667,8.0,0
3,ID_266224270337267278,266224,Kenya,270337,267278,Type_1,6123.0,6123.0,2022-09-29,2022-10-06,7,Repeat Loan,318.8,0.052066,319.0,0
4,ID_250369246185267278,250369,Kenya,246185,267278,Type_1,2953.0,2974.0,2022-08-27,2022-09-03,7,Repeat Loan,15.0,0.005080,15.0,0


In [85]:
dl.head()

,Country,Indicator,YR2001,YR2002,YR2003,YR2004,YR2005,YR2006,YR2007,YR2008,YR2009,YR2010,YR2011,YR2012,YR2013,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023
0,Ghana,"Inflation, consumer prices (annual %)",41.509496,9.360932,29.772980,18.042739,15.438992,11.679184,10.734267,16.494640,19.246948,10.733390,8.728459,11.186341,11.666192,15.489616,17.149970,17.454635,12.371922,7.808765,7.143640,9.887290,9.971089,31.255895,38.106966
1,Cote d'Ivoire,"Inflation, consumer prices (annual %)",4.361529,3.077265,3.296807,1.457988,3.885830,2.467191,1.892006,6.308528,1.019505,1.226456,4.912434,1.304511,2.581170,0.448682,1.251500,0.723178,0.685881,0.359409,-1.106863,2.425007,4.091952,5.276167,4.387117
2,Kenya,"Inflation, consumer prices (annual %)",5.738598,1.961308,9.815691,11.624036,10.312778,14.453734,9.758880,26.239817,9.234126,3.961389,14.022491,9.377770,5.717494,6.878155,6.582154,6.297250,8.005650,4.689806,5.239638,5.405162,6.107936,7.659863,7.671396
3,Ghana,"Official exchange rate (LCU per US$, period av...",0.716305,0.792417,0.866764,0.899495,0.905209,0.915107,0.932619,1.052275,1.404967,1.429983,1.520625,1.824867,1.981350,2.896575,3.714642,3.909817,4.350533,4.585325,5.217367,5.595708,5.805700,8.272400,11.020408
4,Cote d'Ivoire,"Official exchange rate (LCU per US$, period av...",732.397693,693.713226,579.897426,527.338032,527.258363,522.425625,478.633718,446.000041,470.293423,494.794262,471.248626,510.556338,493.899624,493.757330,591.211698,592.605615,580.656750,555.446458,585.911013,575.586005,554.530675,623.759701,606.569750


#### STEP 3: Data Cleaning

In [86]:
# Summary statistics

def check_df(df):
    # Shape of the dataset
    print(f"Dataset Shape: {df.shape}")

    # Column names
    print("\nColumns:")
    print(display(df.columns.tolist()))

    # Data types
    print("\nData Types:")
    display(df.dtypes)

    # Missing values
    print("\nMissing Values:")
    display(df.isnull().sum())

check_df(df) 

Dataset Shape: (68654, 16)

Columns:


['ID',
 'customer_id',
 'country_id',
 'tbl_loan_id',
 'lender_id',
 'loan_type',
 'Total_Amount',
 'Total_Amount_to_Repay',
 'disbursement_date',
 'due_date',
 'duration',
 'New_versus_Repeat',
 'Amount_Funded_By_Lender',
 'Lender_portion_Funded',
 'Lender_portion_to_be_repaid',
 'target']

None

Data Types:


ID                                 str
customer_id                      int64
country_id                         str
tbl_loan_id                      int64
lender_id                        int64
loan_type                          str
Total_Amount                   float64
Total_Amount_to_Repay          float64
disbursement_date                  str
due_date                           str
duration                         int64
New_versus_Repeat                  str
Amount_Funded_By_Lender        float64
Lender_portion_Funded          float64
Lender_portion_to_be_repaid    float64
target                           int64
dtype: object


Missing Values:


ID                             0
customer_id                    0
country_id                     0
tbl_loan_id                    0
lender_id                      0
loan_type                      0
Total_Amount                   0
Total_Amount_to_Repay          0
disbursement_date              0
due_date                       0
duration                       0
New_versus_Repeat              0
Amount_Funded_By_Lender        0
Lender_portion_Funded          0
Lender_portion_to_be_repaid    0
target                         0
dtype: int64

In [87]:
# Check for unique loan_ids
df["tbl_loan_id"].nunique()

66569

In [88]:
# Check for unique customer_ids
df["customer_id"].nunique()

6517

In [89]:
# Show unique lenders per loan_id
df.groupby("tbl_loan_id")["lender_id"].nunique().value_counts().sort_index()

lender_id
1    64484
2     2085
Name: count, dtype: int64

In [90]:
# Target distribution
df["target"].value_counts()

target
0    67396
1     1258
Name: count, dtype: int64

In [91]:
df["target"].value_counts(normalize=True)

target
0    0.981676
1    0.018324
Name: proportion, dtype: float64

In [92]:
# Check defaulters by new vs repeat customers
#.sum() because we are dealing with a binary target variable
df.groupby("New_versus_Repeat")["target"].agg(["count", "sum", "mean"])

,count,sum,mean
New_versus_Repeat,,,
New Loan,542,108,0.199262
Repeat Loan,68112,1150,0.016884


In [93]:
# Check defaulters by loan type
df.groupby("loan_type")["target"].agg(["count", "sum", "mean"])

,count,sum,mean
loan_type,,,
Type_1,61823,550,0.008896
Type_10,45,3,0.066667
Type_11,12,0,0.000000
Type_12,4,2,0.500000
Type_13,10,0,0.000000
Type_14,20,0,0.000000
Type_15,36,0,0.000000
Type_16,4,0,0.000000
Type_17,11,5,0.454545


In [94]:
# Converting date columns to datetime format
df["disbursement_date"] = pd.to_datetime(df["disbursement_date"])
df["due_date"] = pd.to_datetime(df["due_date"])

In [95]:
# Investigating the "financial columns"
df[
    [
        "Total_Amount",
        "Total_Amount_to_Repay",
        "Amount_Funded_By_Lender",
        "Lender_portion_Funded",
        "Lender_portion_to_be_repaid"
    ]
].head(10)

,Total_Amount,Total_Amount_to_Repay,Amount_Funded_By_Lender,Lender_portion_Funded,Lender_portion_to_be_repaid
0,16477.0,16642.0,2436.00,0.147842,2460.0
1,5599.0,5599.0,2799.50,0.500000,2800.0
2,4679.0,4746.0,7.80,0.001667,8.0
3,6123.0,6123.0,318.80,0.052066,319.0
4,2953.0,2974.0,15.00,0.005080,15.0
5,17413.0,17413.0,5223.90,0.300000,5224.0
6,6660.0,6660.0,2.19,0.000329,2.0
7,10465.0,10785.0,599.65,0.057301,618.0
8,6249.0,6249.0,1691.07,0.270614,1691.0
9,57853.0,59262.0,17355.90,0.300000,17779.0


In [96]:
df[
    [
        "Total_Amount",
        "Total_Amount_to_Repay",
        "Amount_Funded_By_Lender",
        "Lender_portion_Funded",
        "Lender_portion_to_be_repaid"
    ]
].describe()

,Total_Amount,Total_Amount_to_Repay,Amount_Funded_By_Lender,Lender_portion_Funded,Lender_portion_to_be_repaid
count,6.865400e+04,6.865400e+04,6.865400e+04,68654.000000,6.865400e+04
mean,1.459270e+04,1.539934e+04,2.501309e+03,0.219135,2.603978e+03
std,1.336684e+05,1.611572e+05,1.148607e+04,0.129656,1.291656e+04
min,5.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
25%,2.289000e+03,2.321000e+03,2.390000e+02,0.121316,2.420000e+02
50%,5.248000e+03,5.299000e+03,9.090000e+02,0.300000,9.270000e+02
75%,1.138800e+04,1.159600e+04,2.258400e+03,0.300000,2.303000e+03
max,2.000000e+07,2.415284e+07,1.600000e+06,1.168119,1.821338e+06


__Note:__
- min Total_Amount > min Total_Amount_to_Repay
- max Lender_portion_Funded = 1.16 instead of 1 

In [97]:
# Check for any rows where Lender_portion_funded > 1
df[df["Lender_portion_Funded"] > 1][
    [
        "Total_Amount",
        "Amount_Funded_By_Lender",
        "Lender_portion_Funded",
        "Lender_portion_to_be_repaid",
        "Total_Amount_to_Repay"
    ]
].head(10)

,Total_Amount,Amount_Funded_By_Lender,Lender_portion_Funded,Lender_portion_to_be_repaid,Total_Amount_to_Repay
37945,2249.0,2627.1,1.168119,2627.0,2249.0


In [98]:
(df["Lender_portion_Funded"] > 1).sum()

np.int64(1)

- One record was found

In [99]:
# Standardardize the column names
df.columns = df.columns.str.lower()

df.columns.to_list()

['id',
 'customer_id',
 'country_id',
 'tbl_loan_id',
 'lender_id',
 'loan_type',
 'total_amount',
 'total_amount_to_repay',
 'disbursement_date',
 'due_date',
 'duration',
 'new_versus_repeat',
 'amount_funded_by_lender',
 'lender_portion_funded',
 'lender_portion_to_be_repaid',
 'target']

In [100]:
df.head()

,id,customer_id,country_id,tbl_loan_id,lender_id,loan_type,total_amount,total_amount_to_repay,disbursement_date,due_date,duration,new_versus_repeat,amount_funded_by_lender,lender_portion_funded,lender_portion_to_be_repaid,target
0,ID_257244237466267278,257244,Kenya,237466,267278,Type_1,16477.0,16642.0,2022-08-15,2022-08-22,7,Repeat Loan,2436.0,0.147842,2460.0,0
1,ID_249619214444267278,249619,Kenya,214444,267278,Type_1,5599.0,5599.0,2022-07-13,2022-07-20,7,Repeat Loan,2799.5,0.500000,2800.0,0
2,ID_251457257838267278,251457,Kenya,257838,267278,Type_1,4679.0,4746.0,2022-09-12,2022-09-19,7,Repeat Loan,7.8,0.001667,8.0,0
3,ID_266224270337267278,266224,Kenya,270337,267278,Type_1,6123.0,6123.0,2022-09-29,2022-10-06,7,Repeat Loan,318.8,0.052066,319.0,0
4,ID_250369246185267278,250369,Kenya,246185,267278,Type_1,2953.0,2974.0,2022-08-27,2022-09-03,7,Repeat Loan,15.0,0.005080,15.0,0


In [101]:
# Save the cleaned dataset 
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(processed_dir / "loans_cleaned.csv", index=False)